# Análise comparativa U-Net vs SegFormer (estágio 12)

Consolida os artefatos persistidos pelo estágio 11 — resumos por modelo/fonte, métricas por patch e predições binárias — em uma comparação estatística emparelhada por patch entre a **U-Net (CNN)** e o **SegFormer (ViT)** na **fonte escolhida manualmente** (`ground_truth.chosen_source`). Para cada métrica de pixel (IoU, F1, Precisão e Recall), calcula a diferença SegFormer − U-Net, intervalo de confiança bootstrap, teste t pareado, teste de Wilcoxon, correção de Holm, tamanhos de efeito e a contagem de patches em que cada modelo vence. Gera também a figura comparativa e os mapas de erro (TP/FP/FN/ignorado) de patches representativos. O estágio não treina nem infere novamente: apenas analisa os artefatos existentes do estágio 11.


## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.


In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/jotap1101/tcc-lab/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")


## Dependências pinadas

Garante as versões fixadas em `requirements-runtime.txt`, incluindo o `scipy` usado nos testes estatísticos, com o mesmo conjunto nas duas plataformas. Quando uma biblioteca com extensão compilada já importada (numpy, scipy, torch, entre outras) diverge da versão pinada, o kernel é reiniciado automaticamente para evitar inconsistência de ABI; nesse caso, o notebook é reexecutado a partir da primeira célula.


In [ ]:
# Garante as versões pinadas e reinicia o kernel quando a ABI carregada divergir.
from src.setup import install_runtime_requirements

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
install_runtime_requirements(requirements)


## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolve todos os caminhos de armazenamento definidos em `src/config.yaml`, incluindo os artefatos do estágio 11, a comparação e as figuras. A fonte analisada é a mesma escolhida manualmente na fase de ground truth.


In [ ]:
# Resolve o contexto de runtime: plataforma, armazenamento canônico e configuração.
from src.comparative import comparison_path, comparative_dir
from src.data.source_decision import chosen_source
from src.setup import initialize

ctx = initialize(workspace)
platform = ctx.platform
storage_paths = ctx.storage_paths
config = ctx.config
source_name = chosen_source()
print(f"Plataforma: {platform}")
print(f"Fonte de treinamento/avaliação (decisão manual): {source_name}")
print(f"Comparação: {comparison_path(storage_paths, source_name)}")
print(f"Diretório da comparação: {comparative_dir(storage_paths, source_name)}")
print(f"Figuras: {storage_paths['artifacts_figures']}")


## Reprodutibilidade

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch, além de registrar as versões dos pacotes principais do ambiente. A comparação é somente leitura e análise estatística; a semente global também alimenta o bootstrap dos intervalos de confiança, garantindo resultados idênticos em reexecuções com os mesmos artefatos.


In [ ]:
# Fixa sementes, flags determinísticas e registra as versões do ambiente.
from src.utils import log_environment, setup_reproducibility

setup_reproducibility(config)
log_environment()


## Dependências do estágio 11

Verifica que os resumos de avaliação dos dois modelos existem e que a comparação está vigente frente ao manifesto (estágio 06), à divisão espacial (estágio 07) e à configuração atual. Confirma também a presença das métricas por patch e das predições binárias de cada dobra, sem executar nova inferência nesta etapa.


In [ ]:
# Verifica os resumos do estágio 11 e a vigência do dataset/config atual.
from src.comparative import load_evaluation_summaries, summaries_are_current
from src.data.patch_generation import manifest_is_current, manifest_path
from src.data.spatial_split import fold_count
from src.evaluation import MODEL_NAMES, evaluation_summary_path
from src.utils import check_dependencies

summaries = load_evaluation_summaries(storage_paths, source_name)
check_dependencies(
    {
        f"Resumo de {model_name} (estágio 11)": evaluation_summary_path(
            storage_paths, model_name, source_name
        )
        for model_name in MODEL_NAMES
    }
)
if not manifest_is_current(storage_paths, source_name):
    raise ValueError("Manifesto desatualizado frente às entradas; execute o estágio 06 antes.")
if not summaries_are_current(storage_paths, source_name, summaries):
    raise ValueError(
        "Artefatos do estágio 11 desatualizados ou incompletos; reexecute a avaliação antes."
    )
print(f"Manifesto: {manifest_path(storage_paths, source_name)}")
print(f"Patches: {summaries['unet']['n_patches']} | Dobras: {fold_count()}")


## Comparação estatística emparelhada

Executa a comparação entre os modelos (`src.comparative.analyze_comparison`). As métricas por patch dos dois modelos são emparelhadas pelo `patch_id` e validam que a referência, a dobra e a proporção de café coincidem. Para cada métrica, calcula a diferença SegFormer − U-Net, o intervalo de confiança bootstrap (2.000 reamostragens), o teste t pareado, o teste não paramétrico de Wilcoxon dos postos sinalizados, a correção de Holm–Bonferroni para as quatro métricas e os tamanhos de efeito (d_z de Cohen e delta de Cliff). O resultado é persistido em `artifacts/metrics/comparative/<fonte>/comparison.json` e reutilizado apenas enquanto os fingerprints e os pesos do estágio 11 coincidem.


In [ ]:
# Consolida a comparação estatística (reutiliza quando o fingerprint do estágio 11 coincide).
from src.comparative import analyze_comparison

analysis = analyze_comparison(storage_paths, source_name=source_name)
comparison = analysis.payload
patch_metrics = analysis.patch_metrics
print(f"Resumo da comparação: {analysis.summary_path}")


## Desempenho agregado por modelo

Apresenta as métricas agregadas *micro* (TP/FP/FN somados entre todas as dobras), a média e o desvio das métricas por dobra e o número de patches avaliados de cada modelo.


In [ ]:
# Monta a tabela de desempenho agregado por modelo.
import pandas as pd

from src.comparative import MODEL_LABELS

rows = []
for model_name in MODEL_NAMES:
    summary = summaries[model_name]
    aggregate = summary["aggregate"]["metrics"]
    rows.append(
        {
            "modelo": MODEL_LABELS[model_name],
            "patches": summary["n_patches"],
            "iou": round(aggregate["iou"], 4),
            "f1": round(aggregate["f1"], 4),
            "precision": round(aggregate["precision"], 4),
            "recall": round(aggregate["recall"], 4),
            "iou_dobra_media": round(summary["fold_mean"]["iou"], 4),
            "iou_dobra_desvio": round(summary["fold_std"]["iou"], 4),
        }
    )
pd.DataFrame(rows)


## Testes estatísticos por métrica

Exibe, para cada métrica de pixel, o número de pares, a diferença média e o intervalo de confiança de 95%, os p-valores do teste t pareado, do Wilcoxon e do Wilcoxon corrigido por Holm, além dos tamanhos de efeito e da contagem de patches em que o SegFormer vence, empata ou perde para a U-Net. Valores indisponíveis (por exemplo, quando não há variabilidade suficiente) aparecem como `n/a`.


In [ ]:
# Monta a tabela dos testes estatísticos emparelhados por métrica.


def _format(value, digits=4):
    if value is None:
        return "n/a"
    return f"{value:.{digits}f}"


rows = []
for metric, stats in comparison["paired"].items():
    rows.append(
        {
            "metrica": metric,
            "pares": stats["n_pairs"],
            "diferenca_media": _format(stats["mean_difference"]),
            "ic95": (
                f"[{_format(stats['ci95_low'])}, {_format(stats['ci95_high'])}]"
            ),
            "p_t": _format(stats["t_pvalue"]),
            "p_wilcoxon": _format(stats["wilcoxon_pvalue"]),
            "p_holm": _format(stats["p_value_holm"]),
            "cohen_dz": _format(stats["cohens_dz"]),
            "cliff_delta": _format(stats["cliffs_delta"]),
            "segformer_melhor": stats["segformer_better"],
            "empates": stats["ties"],
            "unet_melhor": stats["unet_better"],
        }
    )
pd.DataFrame(rows)


## Diferenças por dobra

Mostra as diferenças das métricas agregadas de cada dobra espacial (SegFormer − U-Net), permitindo verificar se o sinal da comparação é consistente entre as regiões do AOI e não apenas um efeito do agregado.


In [ ]:
# Monta a tabela das diferenças por dobra (SegFormer − U-Net).
rows = [
    {
        "dobra": item["fold"],
        "patches": item["n_patches"],
        "iou": round(item["delta"]["iou"], 4),
        "f1": round(item["delta"]["f1"], 4),
        "precision": round(item["delta"]["precision"], 4),
        "recall": round(item["delta"]["recall"], 4),
    }
    for item in comparison["per_fold_delta"]
]
pd.DataFrame(rows)


## Figura comparativa

Renderiza e persiste a figura com a distribuição de IoU por patch, o IoU agregado por dobra, a dispersão entre os modelos (colorida pela proporção de café) e o histograma da diferença de IoU. A figura é regenerada a cada execução para refletir os artefatos vigentes.


In [ ]:
# Renderiza e persiste a figura comparativa no diretório canônico de figuras.
from src.comparative import save_comparison_figure

comparison_figure = save_comparison_figure(
    storage_paths, source_name, comparison, patch_metrics
)


## Seleção dos patches para os mapas de erro

Seleciona, de forma determinística, patches com café que representem a maior e a menor diferença de IoU entre os modelos e o maior e o menor IoU médio. Empates são desfeitos pelo `patch_id`, garantindo a mesma amostra em reexecuções.


In [ ]:
# Seleciona os patches representativos para os mapas de erro.
from src.comparative import select_error_map_patches

selected_patches = select_error_map_patches(patch_metrics)
pd.DataFrame(
    [
        {
            "patch_id": item["patch_id"],
            "caso": item["label"],
            "dobra": item["fold"],
            "iou_unet": round(item["iou_unet"], 4),
            "iou_segformer": round(item["iou_segformer"], 4),
            "diferenca_iou": round(item["iou_difference"], 4),
        }
        for item in selected_patches
    ]
)


## Mapas de erro

Renderiza e persiste, para cada patch selecionado, o RGB (B4/B3/B2), a máscara de referência e os mapas de erro dos dois modelos com a codificação verdadeiro positivo (verde), falso positivo (vermelho), falso negativo (azul), verdadeiro negativo (cinza) e pixels ignorados fora do AOI (branco). A figura é salva em `artifacts/figures/`.


In [ ]:
# Renderiza e persiste os mapas de erro dos patches selecionados.
from src.comparative import save_error_maps_figure

error_maps_figure = save_error_maps_figure(storage_paths, source_name, selected_patches)


## Resumo da etapa

Exibe o resumo da comparação — patches analisados, IoU agregado de cada modelo, diferença agregada, p-valor corrigido do IoU e caminhos dos artefatos persistidos.


In [ ]:
# Exibe o resumo da etapa e os caminhos dos artefatos.
from src.utils import print_summary

unet_aggregate = comparison["models"]["unet"]["aggregate"]["metrics"]
segformer_aggregate = comparison["models"]["segformer"]["aggregate"]["metrics"]
summary = {
    "Fonte": source_name,
    "Patches comparados": comparison["n_patches"],
    "IoU agregado U-Net": f"{unet_aggregate['iou']:.4f}",
    "IoU agregado SegFormer": f"{segformer_aggregate['iou']:.4f}",
    "Delta IoU (SegFormer - U-Net)": f"{comparison['aggregate_delta']['iou']:+.4f}",
    "p-valor Wilcoxon do IoU (Holm)": _format(
        comparison["paired"]["iou"]["p_value_holm"]
    ),
    "Comparacao": str(analysis.summary_path),
    "Figura comparativa": str(comparison_figure),
    "Mapas de erro": str(error_maps_figure),
}
print_summary(summary, "12")
